# Lesson 1.5 — Logging & Structured Logging

**Level: Beginner → Intermediate**

## Learning Objectives
- Configure Python's `logging` module for DevOps scripts
- Use `structlog` to emit JSON-structured log entries
- Bind persistent context (service, env, trace_id) to every log entry
- Understand why structured logging is required by CloudWatch, ELK, and Loki
- Write a reusable `logger.py` module

**Produces:** `logger.py` — a portable structured-logging module.

*Official refs:*
- *[Python logging HOWTO](https://docs.python.org/3/howto/logging.html)*
- *[structlog documentation](https://www.structlog.org/en/stable/)*

## Concepts

### Why Structured Logging?

**Unstructured log** (human-readable, machine-unfriendly):
```
2025-01-15 10:23 ERROR Failed to connect to database after 3 retries
```

**Structured (JSON) log** (machine-parseable by every log aggregator):
```json
{"timestamp": "2025-01-15T10:23:00Z", "level": "error",
 "event": "db_connection_failed", "retries": 3,
 "service": "api", "env": "production", "trace_id": "abc-123"}
```

With structured logs you can:
- **Filter**: `level=error AND service=api`
- **Aggregate**: count errors per service per hour in CloudWatch Insights
- **Alert**: trigger PagerDuty when `retries > 5`
- **Trace**: correlate logs across services with `trace_id`

### structlog Processor Pipeline

```
log.info("event", key=value)
      ↓
[stdlib.add_log_level]   → adds "level": "info"
[processors.TimeStamper] → adds "timestamp": "2025-..."
[contextvars.merge]      → merges bound context vars
[JSONRenderer]           → serialises to JSON string
      ↓
output
```

*Reference: [structlog docs — processors](https://www.structlog.org/en/stable/processors.html)*

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "structlog", "--quiet"],
               capture_output=True)
import structlog, logging
print(f"structlog version: {structlog.__version__}")

## Code-Along: Step 1 — stdlib logging

In [ ]:
# Python stdlib logging — foundational knowledge
# Reference: https://docs.python.org/3/howto/logging.html

def configure_stdlib_logging(level: str = "INFO") -> logging.Logger:
    """Configure the root logger with a timestamp formatter."""
    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%SZ"
    )
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(fmt)
    logging.basicConfig(
        level=getattr(logging, level.upper(), logging.INFO),
        handlers=[handler],
        force=True,
    )
    return logging.getLogger("ops")

logger = configure_stdlib_logging("DEBUG")
logger.debug("checking_credentials")
logger.info("inventory_scan_started — region=us-east-1")
logger.warning("rate_limit_approaching — requests_remaining=50")
logger.error("ec2_api_error — region=eu-west-1 — AccessDenied")

## Code-Along: Step 2 — structlog with JSON Output

In [ ]:
import os

# Configure structlog once at startup
def configure_structlog(level: str = "INFO", environment: str = "development") -> None:
    """Set up structlog processor pipeline.

    Reference: https://www.structlog.org/en/stable/getting-started.html
    """
    processors = [
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.set_exc_info,  # adds traceback to log on exception
    ]
    # Dev: colored, human-readable.  Production: compact JSON.
    if environment in ("production", "staging", "ci"):
        processors.append(structlog.processors.JSONRenderer())
    else:
        processors.append(structlog.dev.ConsoleRenderer())

    structlog.configure(
        processors=processors,
        wrapper_class=structlog.stdlib.BoundLogger,
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
    )

os.environ["ENVIRONMENT"] = "development"  # pretty output in the notebook
configure_structlog(environment="development")

log = structlog.get_logger()
log.info("deployment_started", service="web-api", tag="v1.2.3", env="staging")
log.warning("high_cpu", instance_id="i-0abc123", cpu_percent=82.5)
log.error("connection_failed", host="db.internal", port=5432, retries=3)

## Code-Along: Step 3 — Bound Loggers with Persistent Context

In [ ]:
import uuid

# Bind service-level context once; it appears in every log entry
service_log = structlog.get_logger().bind(
    service="ops-automation-platform",
    environment="production",
    version="1.0.0",
)

def run_ec2_inventory(region: str, trace_id: str | None = None) -> None:
    """Simulate inventory with structured trace-correlated logging."""
    op_log = service_log.bind(
        operation="ec2_inventory",
        region=region,
        trace_id=trace_id or str(uuid.uuid4())[:8],
    )
    op_log.info("scan_started")
    op_log.info("instances_found", count=5, running=4, stopped=1)
    op_log.warning("stopped_instance_found",
                   instance_id="i-0abc123", stopped_since="2025-01-10")
    op_log.info("scan_completed", duration_ms=142)

run_ec2_inventory("us-east-1")
print()
run_ec2_inventory("eu-west-1", trace_id="req-xyz-999")

## Code-Along: Step 4 — Write the Reusable Logger Module

In [ ]:
%%writefile logger.py
"""Reusable structured logging module for ops-automation-platform.

Usage:
    from logger import configure_logging, get_logger

    configure_logging()                            # call once at startup
    log = get_logger("my_module", env="production")
    log.info("task_started", task_id="abc-123")

Reference: https://www.structlog.org/en/stable/
"""
import os
import structlog

def configure_logging(level: str | None = None) -> None:
    """Configure structlog processor pipeline from environment variables."""
    environment = os.environ.get("ENVIRONMENT", "development")
    processors = [
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.set_exc_info,
    ]
    if environment in ("production", "staging", "ci"):
        processors.append(structlog.processors.JSONRenderer())
    else:
        processors.append(structlog.dev.ConsoleRenderer())

    structlog.configure(
        processors=processors,
        wrapper_class=structlog.stdlib.BoundLogger,
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
    )

def get_logger(name: str = "ops", **context) -> structlog.stdlib.BoundLogger:
    """Get a bound logger with the given persistent context fields."""
    return structlog.get_logger(name).bind(**context)


In [ ]:
from logger import configure_logging, get_logger

configure_logging()
log = get_logger("ec2_module", service="ops-platform", env="development")

log.debug("checking_aws_credentials")
log.info("scan_started", region="us-east-1")

try:
    raise ConnectionError("Endpoint unreachable")
except ConnectionError:
    log.error("aws_connection_failed", region="us-east-1")

## Exercise

Extend `logger.py` to optionally write JSON logs to a file.

**Your task:**
1. Add `log_file: str | None = None` to `configure_logging()`
2. When provided, also write all logs to the file in JSON format
3. Verify both stdout and the file receive the same entries

**Hint:** Integrate with Python's `logging` module. See
[structlog stdlib integration](https://www.structlog.org/en/stable/standard-library.html).

In [ ]:
# TODO: modify configure_logging() to accept log_file=
# TODO: call configure_logging(log_file="ops.log")
# TODO: write a few log entries
# TODO: read and print ops.log to confirm JSON format


## Key Takeaways
- JSON logs are the production standard — CloudWatch Insights, ELK, and Loki all ingest JSON natively
- `structlog.get_logger().bind(...)` attaches persistent context once, appears in every line
- Always set log level from `LOG_LEVEL` env var — never hardcode it
- The `logger.py` module from this lesson is imported by every Phase 2–5 lesson

---
**Phase 1 Complete!**

**Next Phase:** [Phase 2 — Cloud Automation (CloudOps)](../phase-2-cloud-automation-cloudops/lesson-2.1-boto3-basics.ipynb)